In [1]:
%reload_ext autoreload


In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import numpy as np
from typing import Generator, Any
import pandas as pd
import json
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
import os
from rich import print as rprint
from mbay_nmt.utils import domain as d
from mbay_nmt.utils.models import new_object_id
from datasets import load_dataset, Dataset, DatasetDict
from rich import print as rprint

load_dotenv()

True

In [4]:
# from huggingface_hub import notebook_login

# notebook_login()

## Huggingface datasets creation


In [5]:
CSV_DATASET_PATH = "../../datasets/mbay-translations-review.csv"
SPLIT_DATASET_PATH = "../../datasets/mbay-translations/"
TOKENIZED_DATASET_PATH = lambda x: f"../../datasets/mbay-translations-tokenized/{x}/"

In [6]:
# df = pd.read_csv(CSV_DATASET_PATH, compression="gzip")

In [7]:
base_dst = Dataset.from_csv(CSV_DATASET_PATH)
base_dst

Dataset({
    features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english', 'edited'],
    num_rows: 10732
})

In [8]:
# dst = Dataset.from_pandas(df)
# dst = dst.remove_columns(["__index_level_0__"])
train_test_valid_dst = DatasetDict.load_from_disk(SPLIT_DATASET_PATH)
train_test_valid_dst

DatasetDict({
    train: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 8580
    })
    test: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
    validation: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
})

## NLLB

[StackOverflow: How to add new language to NLLB tokenizer in HuggingFace](https://stackoverflow.com/questions/76199337/how-to-add-new-language-to-nllb-tokenizer-in-huggingface)

### Plan:

Tokenizer appears to be missing a few tokens for Mbay so we're probably better off continuing training the tokenizer.

What we will do:

1. Add the mbay src language token.
2. Continue training on the unsplit dataset.
3. Save + push to HF hub.


In [9]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
# model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

In [10]:
new_special_tokens = nllb_tokenizer.additional_special_tokens + ["mbay_Latn"]
nllb_tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})

1

In [11]:
print("mbay_Latn" in nllb_tokenizer.additional_special_tokens)

True


In [12]:
lang_code = "mbay_Latn"
[token_id] = nllb_tokenizer.convert_tokens_to_ids([lang_code])
rprint(lang_code, token_id)
nllb_tokenizer.lang_code_to_id[lang_code] = token_id
if hasattr(nllb_tokenizer, "id_to_lang_code"):
    nllb_tokenizer.id_to_lang_code[token_id] = lang_code

mbay_Latn 256204

In [13]:
nllb_tokenizer.lang_code_to_id[lang_code]

256204

In [14]:
nllb_tokenizer.is_fast

True

In [15]:
nllb_tokenizer.src_lang = lang_code

In [16]:
texts = base_dst["mbay"][:10]
rprint(texts)
tokens = nllb_tokenizer(texts)
# print(tokens["input_ids"])
recovered = nllb_tokenizer.batch_decode(tokens["input_ids"])
rprint(recovered)

[
    'àlmbétɨ̀, àlmétɨ̀',
    'gà àlmbétɨ̀',
    'ī-gá àlmbétɨ̀ ādɨ̄-m̄.',
    'kùm-àlmbétɨ̀',
    'Màn̄ à ɔ̀dɨ̀ kùm-àlmbétɨ̀ ànḛ̄ à ùnjɨ̄ àĺ.',
    'kāgɨ̄-àlmbétɨ̀',
    'ādɨ̄-m̄ kāgɨ̄-àlmbétɨ̀ kɨ́rā m̄-ɗāa-ň mbī-ḿ.',
    'ámpɨ̀rmē',
    'ámpɨ̀rmē ɔ̀sɨ̀-á kɨ̀ kḭ̄ḭ.',
    'àngérì'
]

[
    'mbay_Latn àlmbétɨ̀, àlmétɨ̀</s>',
    'mbay_Latn gà àlmbétɨ̀</s>',
    'mbay_Latn ī-gá àlmbétɨ̀ ādɨ̄-m̄.</s>',
    'mbay_Latn kùm-àlmbétɨ̀</s>',
    'mbay_Latn Màn̄ à ɔ̀dɨ̀ kùm-àlmbétɨ̀ ànḛ̄ à ùnjɨ̄ àĺ.</s>',
    'mbay_Latn kāgɨ̄-àlmbétɨ̀</s>',
    'mbay_Latn ādɨ̄-m̄ kāgɨ̄-àlmbétɨ̀ kɨ́rā m̄-ɗāa-ň mbī-ḿ.</s>',
    'mbay_Latn ámpɨ̀rmē</s>',
    'mbay_Latn ámpɨ̀rmē ɔ̀sɨ̀-á kɨ̀ kḭ̄<unk>.</s>',
    'mbay_Latn àngérì</s>'
]

In [25]:
nllb_tokenizer.tokenize(src)

['▁Dè',
 'ē',
 '-',
 'k',
 'ɨ',
 '́',
 '-',
 'dè',
 '̰',
 'é',
 '̰',
 '▁nd',
 'ì',
 '▁kūr',
 'ō',
 '▁mb',
 'ùr',
 '̄',
 '▁g',
 '<unk>',
 '̀',
 'rì',
 '̰',
 '-',
 'g',
 '<unk>',
 '̀',
 'rì',
 '̰',
 '.']

In [22]:
src = "Dèē-kɨ́-dḛ̀ḛ́ ndì kūrō mbùr̄ gṵ̀rḭ̀-gṵ̀rḭ̀."
corrupted = "mbay_Latn Dèē-kɨ́-dḛ̀ḛ́ ndì kūrō mbùr̄ g<unk>̀rḭ̀-g<unk>̀rḭ̀."

In [ ]:
tokenizer.

In [98]:

tokenizer = nllb_tokenizer
input_texts = [src]

import re


def find_unknown_substrings(tokenizer, input_texts, verbose=False):
    unk_substrings = set()
    for input_text in input_texts:
        encoded = tokenizer.encode(input_text, add_special_tokens=False)
        tokens = tokenizer.convert_ids_to_tokens(encoded)
        decoded = tokenizer.decode(encoded)

        # Find the UNK tokens and their corresponding substrings in the input text
        last_unk_idx = 0
        # all multi whitespace is replaced by single space
        suffix =  re.sub(r'\s+', ' ', input_text)
        # suffix = input_text
        for i, (token, id) in enumerate(zip(tokens, encoded)):
            if token == tokenizer.unk_token:
                # Find the substring by looking at the surrounding tokens
                prefix = tokenizer.convert_tokens_to_string(tokens[last_unk_idx:i])
                next_token = tokenizer.convert_tokens_to_string(tokens[i + 1 : i + 2])

                truncated = suffix.removeprefix(prefix)
                suffix_start = truncated.find(next_token)
                if suffix_start == 0:
                    # This is a single token
                    unk_subtring = truncated
                else:
                    unk_subtring = truncated[:suffix_start]
                if verbose and unk_subtring not in unk_substrings:
                    rprint(f"{input_text=}, {decoded=}, {prefix=}, {next_token=}, {truncated=}, {unk_subtring=}")
                unk_substrings.add(unk_subtring)
                last_unk_idx = i + 1
                suffix = truncated[suffix_start:]

    return unk_substrings


#     # Output the substrings that are tokenized as UNK
# print(f"Substrings tokenized as UNK in '{input_text}': {unk_substrings}")

In [100]:
find_unknown_substrings(tokenizer, base_dst["mbay"], verbose=True)

input_text='ámpɨ̀rmē ɔ̀sɨ̀-á kɨ̀ kḭ̄ḭ.', decoded='ámpɨ̀rmē ɔ̀sɨ̀-á kɨ̀ kḭ̄<unk>.', prefix='ámpɨ̀rmē ɔ̀sɨ̀-á kɨ̀ kḭ̄',
next_token='.', truncated='ḭ.', unk_subtring='ḭ'

input_text='Dà-būu-lòo ì lòo kɨ́ dèē-kɨ́-dḛ̀ḛ́-gɨ̄ à ṵ̀ṵ̄-n búu-hòr tɨ́.', decoded='Dà-būu-lòo ì lòo kɨ́ dèē-kɨ́-dḛ̀ḛ́-gɨ̄ à
<unk>̀<unk>̄-n búu-hòr tɨ́.', prefix='Dà-būu-lòo ì lòo kɨ́ dèē-kɨ́-dḛ̀ḛ́-gɨ̄ à ', next_token='̀', truncated='ṵ̀ṵ̄-n búu-hòr
tɨ́.', unk_subtring='ṵ'

input_text='Dàngá̰ỳ-gɨ̄ ndɨ̀ ùr-n̄ hól-ɓēe kḭ̀ḭ lò kòmàndá.', decoded='Dàngá̰ỳ-gɨ̄ ndɨ̀ ùr-n̄ hól-ɓēe kḭ̀<unk> lò kòmàndá.',
prefix='Dàngá̰ỳ-gɨ̄ ndɨ̀ ùr-n̄ hól-ɓēe kḭ̀', next_token='lò', truncated='ḭ lò kòmàndá.', unk_subtring='ḭ '

input_text='kɨ́-mḛ̄ḛ-dɔ̀gɨ̀', decoded='kɨ́-mḛ̄<unk>-dɔ̀gɨ̀', prefix='kɨ́-mḛ̄', next_token='-', truncated='ḛ-dɔ̀gɨ̀',
unk_subtring='ḛ'

input_text='ī ngè-ngḛ̀ḛ ngá̰y; ngōn-kó̰o̰-í tā-ànḛ̄ à ɗāa kɨ̀là.', decoded='ī ngè-ngḛ̀<unk> ngá̰y; ngōn-kó̰o̰-í tā-ànḛ̄ à ɗāa
kɨ̀là.', prefix='ī ngè-ngḛ̀', next_token='ng', truncated='ḛ ngá̰y; ngōn-kó̰o̰-í tā-ànḛ̄ à ɗāa kɨ̀là.', unk_subtring='ḛ '

...

input_text='╫tɨ̄ gà̰a̰-tà-í nà̰ wɨ́tɨ́ à òtɨ̀-m̄.', decoded='<unk>tɨ̄ gà̰a̰-tà-í nà̰ wɨ́tɨ́ à òtɨ̀-m̄.', prefix='', next_token='t',
truncated='╫tɨ̄ gà̰a̰-tà-í nà̰ wɨ́tɨ́ à òtɨ̀-m̄.', unk_subtring='╫'

input_text='Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-ǹ tɨ́ mbṵ̀y.', decoded='Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-ǹ tɨ́ mb<unk>̀y.', 
prefix='Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-ǹ tɨ́ mb', next_token='̀', truncated='Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-ǹ tɨ́ mbṵ̀y.',
unk_subtring='Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-n'

input_text='lɨ̀bò ì yèl̄ kɨ́ bòo ngá̰y.  à tér rō-ǹ kàm mṵ̀ṵ tɨ́.', decoded='lɨ̀bò ì yèl̄ kɨ́ bòo ngá̰y. à tér rō-ǹ kàm
m<unk>̀<unk> tɨ́.', prefix='lɨ̀bò ì yèl̄ kɨ́ bòo ngá̰y. à tér rō-ǹ kàm m', next_token='̀', truncated='lɨ̀bò ì yèl̄ kɨ́ bòo
ngá̰y. à tér rō-ǹ kàm mṵ̀ṵ tɨ́.', unk_subtring='lɨ'

input_text='lɨ̀bò ì yèl̄ kɨ́ bòo ngá̰y.  à tér rō-ǹ kàm mṵ̀ṵ tɨ́.', decoded='lɨ̀bò ì yèl̄ kɨ́ bòo ngá̰y. à tér rō-ǹ kàm
m<unk>̀<unk> tɨ́.', prefix='̀', next_token='t', truncated='bò ì yèl̄ kɨ́ bòo ngá̰y. à tér rō-ǹ kàm mṵ̀ṵ tɨ́.', 
unk_subtring='bò ì yèl̄ kɨ́ bòo ngá̰y. à '

{' ',
 ' dúu mṵ̀ṵ ',
 ' mṵ̀ṵ',
 '-rō-tàa-gɨ',
 'Bàtú yā rō-n',
 'Ngōn-bḭ̄yā̰ tèē tà gír kó̰o̰-n',
 'Nàr lò-í ì ',
 'bò ì yèl̄ kɨ́ bòo ngá̰y. à ',
 'būlūm ì yèl̄ kɨ́ bòo ngá̰y, à ɓày kɨ̀ bɔ̄gɨ̄-ǹ àĺ. Yùbɨ̄ būlūm ì kḭ̄ḭ ',
 'dí¯ ì yá̰a̰ kɨ́ ngé-kàdɨ',
 'lɨ',
 'm-ɔ́sɨ́ tú¯ m-ṵ',
 'nìbàkíǹ ì kḭ̄ḭ ',
 'sùlú¯ ì yá̰a̰ láā bíl̄-bìl kɨ́ à ɗāa-n̄ kɨ',
 '¯ḿ¯ màjɨ̀, m-ḭ̄ḭ ',
 'àspèríǹ ì kḭ̄ḭ ',
 'ōtɨ́ dà lòo ń bàbá yā rō-n',
 'ɔ̀gɨ̄ bènyē ngōn-ǹ kɨ̀-sḭ́ḭ',
 'ɔ̀r̄ tà ',
 'ḛ',
 'ḛ ',
 'ḛḛ',
 'ḭ',
 'ḭ ',
 'ḭ ādɨ̄ bɔ̀ɔ̄-í.',
 'ḭ ādɨ̄-m̄ sɨ́wé.',
 'ḭḭ',
 'ṵ',
 'ṵ ',
 'ṵ ādɨ̄-á sà mákɨ́-mákɨ́.',
 'ṵṵ',
 'ẁ',
 'ẁ ',
 '╫'}

In [49]:
i = 25
prefix = tokenizer.convert_tokens_to_string(tokens[:i])

next_token = tokenizer.convert_tokens_to_string(tokens[i + 1 : i + 2])

rprint(prefix)
rprint(next_token)

Dèē-kɨ́-dḛ̀ḛ́ ndì kūrō mbùr̄ g<unk>̀rḭ̀-g

̀

In [46]:
truncated = input_text.removeprefix(prefix)
suffix_start = truncated.find(next_token)
truncated[:suffix_start]

'ṵ'

In [49]:
nllb_tokenizer.train_new_from_iterator?

Signature:
nllb_tokenizer.train_new_from_iterator(
    text_iterator,
    vocab_size,
    length=None,
    new_special_tokens=None,
    special_tokens_map=None,
    **kwargs,
)
Docstring:
Trains a tokenizer on a new corpus with the same defaults (in terms of special tokens or tokenization pipeline)
as the current one.

Args:
    text_iterator (generator of `List[str]`):
        The training corpus. Should be a generator of batches of texts, for instance a list of lists of texts
        if you have everything in memory.
    vocab_size (`int`):
        The size of the vocabulary you want for your tokenizer.
    length (`int`, *optional*):
        The total number of sequences in the iterator. This is used to provide meaningful progress tracking
    new_special_tokens (list of `str` or `AddedToken`, *optional*):
        A list of new special tokens to add to the tokenizer you are training.
    special_tokens_map (`Dict[str, str]`, *optional*):
        If you want to rename some of the spe

In [56]:
nllb_tokenizer.add_tokens?

Signature:
nllb_tokenizer.add_tokens(
    new_tokens: Union[str, tokenizers.AddedToken, List[Union[str, tokenizers.AddedToken]]],
    special_tokens: bool = False,
) -> int
Docstring:
Add a list of new tokens to the tokenizer class. If the new tokens are not in the vocabulary, they are added to
it with indices starting from length of the current vocabulary and and will be isolated before the tokenization
algorithm is applied. Added tokens and tokens from the vocabulary of the tokenization algorithm are therefore
not treated in the same way.

Note, when adding new tokens to the vocabulary, you should make sure to also resize the token embedding matrix
of the model so that its embedding matrix matches the tokenizer.

In order to do that, please use the [`~PreTrainedModel.resize_token_embeddings`] method.

Args:
    new_tokens (`str`, `tokenizers.AddedToken` or a list of *str* or `tokenizers.AddedToken`):
        Tokens are only added if they are not already in the vocabulary. `tokenizers

In [54]:
rprint([x for x in dir(nllb_tokenizer) if not x.startswith("_")])

[
    'SPECIAL_TOKENS_ATTRIBUTES',
    'add_special_tokens',
    'add_tokens',
    'added_tokens_decoder',
    'added_tokens_encoder',
    'additional_special_tokens',
    'additional_special_tokens_ids',
    'all_special_ids',
    'all_special_tokens',
    'all_special_tokens_extended',
    'apply_chat_template',
    'as_target_tokenizer',
    'backend_tokenizer',
    'batch_decode',
    'batch_encode_plus',
    'bos_token',
    'bos_token_id',
    'build_inputs_with_special_tokens',
    'can_save_slow_tokenizer',
    'chat_template',
    'clean_up_tokenization',
    'clean_up_tokenization_spaces',
    'cls_token',
    'cls_token_id',
    'convert_added_tokens',
    'convert_ids_to_tokens',
    'convert_tokens_to_ids',
    'convert_tokens_to_string',
    'create_token_type_ids_from_sequences',
    'cur_lang_code',
    'decode',
    'decoder',
    'default_chat_template',
    'deprecation_warnings',
    'encode',
    'encode_plus',
    'eos_token',
    'eos_token_id',
    'from_pretrained',
    'get_added_vocab',
    'get_special_tokens_mask',
    'get_vocab',
    'init_inputs',
    'init_kwargs',
    'is_fast',
    'lang_code_to_id',
    'legacy_behaviour',
    'mask_token',
    'mask_token_id',
    'max_len_sentences_pair',
    'max_len_single_sentence',
    'max_model_input_sizes',
    'model_input_names',
    'model_max_length',
    'name_or_path',
    'num_special_tokens_to_add',
    'pad',
    'pad_token',
    'pad_token_id',
    'pad_token_type_id',
    'padding_side',
    'prefix_tokens',
    'prepare_for_model',
    'prepare_seq2seq_batch',
    'pretrained_init_configuration',
    'pretrained_vocab_files_map',
    'push_to_hub',
    'register_for_auto_class',
    'sanitize_special_tokens',
    'save_pretrained',
    'save_vocabulary',
    'sep_token',
    'sep_token_id',
    'set_src_lang_special_tokens',
    'set_tgt_lang_special_tokens',
    'set_truncation_and_padding',
    'slow_tokenizer_class',
    'special_tokens_map',
    'special_tokens_map_extended',
    'split_special_tokens',
    'src_lang',
    'suffix_tokens',
    'tgt_lang',
    'tokenize',
    'train_new_from_iterator',
    'truncate_sequences',
    'truncation_side',
    'unk_token',
    'unk_token_id',
    'verbose',
    'vocab',
    'vocab_file',
    'vocab_files_names',
    'vocab_size'
]

In [51]:
retrained_nllb_tokenizer = nllb_tokenizer.train_from_iterator(
    base_dst["mbay"], vocab_size=260000
)

AttributeError: 'NllbTokenizerFast' object has no attribute 'train_from_iterator'

In [45]:
retrained_nllb_tokenizer

NllbTokenizerFast(name_or_path='facebook/nllb-200-distilled-600M', vocab_size=17438, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['ace_Arab', 'ace_Latn', 'acm_Arab', 'acq_Arab', 'aeb_Arab', 'afr_Latn', 'ajp_Arab', 'aka_Latn', 'amh_Ethi', 'apc_Arab', 'arb_Arab', 'ars_Arab', 'ary_Arab', 'arz_Arab', 'asm_Beng', 'ast_Latn', 'awa_Deva', 'ayr_Latn', 'azb_Arab', 'azj_Latn', 'bak_Cyrl', 'bam_Latn', 'ban_Latn', 'bel_Cyrl', 'bem_Latn', 'ben_Beng', 'bho_Deva', 'bjn_Arab', 'bjn_Latn', 'bod_Tibt', 'bos_Latn', 'bug_Latn', 'bul_Cyrl', 'cat_Latn', 'ceb_Latn', 'ces_Latn', 'cjk_Latn', 'ckb_Arab', 'crh_Latn', 'cym_Latn', 'dan_Latn', 'deu_Latn', 'dik_Latn', 'dyu_Latn', 'dzo_Tibt', 'ell_Grek', 'eng_Latn', 'epo_Latn', 'est_Latn', 'eus_Latn', 'ewe_Latn', 'fao_Latn', 'pes_Arab',

In [16]:
from tokenizers.trainers import BpeTrainer

trainer = BpeTrainer()

In [11]:
dst_pairs = dst.map(
    format_prompt_output_pairs, batched=True, remove_columns=dst.column_names
)

Map:   0%|          | 0/10726 [00:00<?, ? examples/s]

In [13]:
import random

In [14]:
random.choices(range(len(dst_pairs)), k=10)

[37644, 8011, 18099, 31062, 41935, 11592, 37806, 20706, 23798, 12510]

In [15]:
sample_ids = random.choices(range(len(dst_pairs)), k=10)
dst_pairs[sample_ids]

{'inputs': ['Translate English to Mbay: soon, in a little while',
  'Translate English to Mbay: thoroughly (shrink, thin).',
  "Translate English to Mbay: go announce [s.o.'s] death",
  'Translate English to Mbay: maternal cousin',
  'Translate Mbay to French: kàdɨ̄',
  'Translate English to Mbay: A tradition is a custom of the people.',
  'Translate Mbay to French: hōŕ gìndɨ̄',
  'Translate Mbay to French: ngè-ndò̰o̰ à èl-m̄ tà kɨ́-dà mò̰y tɨ́ lò-ḿ.',
  'Translate French to Mbay: Ils ont mis des entraves aux chevilles du cheval.',
  'Translate Mbay to French: Dèē-kɨ́-dḛ̀ḛ́ kɨ́ ngàw kòy ndūr pútɨ́-pútɨ́.'],
 'targets': ['sḭ́ḭ-kɨ́-nòó-tɨ́',
  'mbḭ́yá̰',
  'àw̄ kɨ̀là-yòo lò [dèē]',
  'ngōn-nān',
  'donner',
  'Yá̰a̰ kōo-wēe ì yá̰a̰ gír lò dèē.',
  'morceau de tuyau en métal',
  'La voyante va me parler de ma maladie (expliquer sa cause).',
  'Lā-n̄ dálā kɔ́ɔ́-njà síndá tɨ́.',
  'La femme dont le mari est décédé se roula énergiquement dans le chagrin.']}

In [21]:
from functools import partial

tokenized_dst_check = dst.map(
    partial(preprocess_records, t5_tokenizer),
    batched=True,
    remove_columns=dst.column_names,
)

Map:   0%|          | 0/10726 [00:00<?, ? examples/s]

In [27]:
tokenized_dst_check

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 42904
})

In [66]:
def detokenize_batch(tokenizer: AutoTokenizer, batch: dict[str, list[list[int]]]):
    input_ids = batch["input_ids"]
    input_ids = np.where(input_ids != -100, input_ids, tokenizer.pad_token_id)
    decoded_input_ids = tokenizer.batch_decode(input_ids, skip_special_tokens=True)

    # labels = batch["labels"]
    # labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    return {
        "inputs": decoded_input_ids,
        # "labels": decoded_labels,
    }

In [67]:
detokenized_dst_check = tokenized_dst_check.map(
    partial(detokenize_batch, t5_tokenizer),
    batched=True,
    # remove_columns=dst.column_names,
)

Map:   0%|          | 0/42904 [00:00<?, ? examples/s]

In [69]:
detokenized_dst_check[sample_ids]["inputs"]

['Translate English to Mbay: soon, in a little while',
 'Translate English to Mbay: thoroughly (shrink, thin).',
 "Translate English to Mbay: go announce [s.o.'s] death",
 'Translate English to Mbay: maternal cousin',
 'Translate Mbay to French: kàdɨ̄',
 'Translate English to Mbay: A tradition is a custom of the people.',
 'Translate Mbay to French: hōŕ gìndɨ̄',
 'Translate Mbay to French: ngè-ndò̰o̰ à èl-m̄ tà kɨ́-dà mò̰y tɨ́ lò-ḿ.',
 'Translate French to Mbay: Ils ont mis des entraves aux chevilles du cheval.',
 'Translate Mbay to French: Dèē-kɨ́-dḛ̀ḛ́ kɨ́ ngàw kòy ndūr pútɨ́-pútɨ́.']

### Final dataset


In [43]:
train_test = dst.train_test_split(0.2)
test_valid = train_test["test"].train_test_split(0.5)

train_test_valid_dst = DatasetDict(
    {
        "train": train_test["train"],
        "test": test_valid["test"],
        "validation": test_valid["train"],
    }
)
train_test_valid_dst

DatasetDict({
    train: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 8580
    })
    test: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
    validation: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
})

In [44]:
train_test_valid_dst["test"][2]

{'id': '64fc50f7286b18ef7de0bfb5',
 'entry_id': '',
 'type': 'entry',
 'mbay': 'ɓà-kùm-dàm-á',
 'french': 'personne aux grands yeux',
 'english': 'person with big eyes'}

In [21]:
train_test_valid_dst = DatasetDict.load_from_disk(SPLIT_DATASET_PATH)

In [22]:
train_test_valid_dst

DatasetDict({
    train: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 8580
    })
    test: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
    validation: Dataset({
        features: ['id', 'entry_id', 'type', 'mbay', 'french', 'english'],
        num_rows: 1073
    })
})

In [15]:
from transformers import AutoTokenizer

t5_tokenizer = AutoTokenizer.from_pretrained("google/byt5-large")

In [16]:
train_test_valid_dst.column_names["train"]

['input_ids', 'attention_mask', 'labels']

In [23]:
from functools import partial
from mbay_nmt.fine_tune_mt5.utils import preprocess_records

final_dst = train_test_valid_dst.map(
    partial(preprocess_records, t5_tokenizer),
    batched=True,
    remove_columns=train_test_valid_dst["train"].column_names,
)

Map:   0%|          | 0/8580 [00:00<?, ? examples/s]

Map:   0%|          | 0/1073 [00:00<?, ? examples/s]

Map:   0%|          | 0/1073 [00:00<?, ? examples/s]

In [26]:
final_dst.save_to_disk(TOKENIZED_DATASET_PATH("nllb"))

Saving the dataset (0/1 shards):   0%|          | 0/34320 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4292 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4292 [00:00<?, ? examples/s]

## M2M100


In [22]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

m2m100_tokenizer = AutoTokenizer.from_pretrained("facebook/m2m100_1.2B")
# model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

In [23]:
lang_code = "mbay_Latn"
[token_id] = nllb_tokenizer.convert_tokens_to_ids([lang_code])
rprint(lang_code, token_id)
nllb_tokenizer.lang_code_to_id[lang_code] = token_id
if hasattr(nllb_tokenizer, "id_to_lang_code"):
    nllb_tokenizer.id_to_lang_code[token_id] = lang_code

mbay_Latn 3

In [24]:
texts = train_test_valid_dst["train"]["mbay"][:3]
rprint(texts)
tokens = m2m100_tokenizer(texts)
print(tokens["input_ids"])
recovered = m2m100_tokenizer.batch_decode(tokens["input_ids"])
rprint(recovered)

['Dèē-kɨ́-dḛ̀ḛ́ ndì kūrō mbùr̄ gṵ̀rḭ̀-gṵ̀rḭ̀.', 'ɗēr', 'kɔ́tɨ́']

[[128022, 81486, 1631, 7, 106, 124748, 2309, 7, 55665, 3, 197, 3, 3229, 2950, 75235, 30979, 2597, 2464, 131, 117394, 230, 3, 1989, 131, 2950, 3, 7, 160, 3, 1989, 131, 2950, 3, 5, 2], [128022, 19010, 9977, 2], [128022, 225, 21185, 2309, 88, 124748, 2309, 2]]


['__en__ Dèē-kɨ́-dè<unk>é<unk>ndì kūrō mbùr̄ g<unk>̀rì<unk>-g<unk>̀rì<unk>.</s>', '__en__ ɗēr</s>', '__en__ kɔ́tɨ́</s>']

In [ ]:
final_dst.save_to_disk(TOKENIZED_DATASET_PATH("m2m100"))